In [ ]:
import pandas as pd
import numpy as np
import pkg_resources
import importlib.metadata
from scipy.stats import ttest_rel
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, log_loss
from sklearn.model_selection import train_test_split, KFold
from sklearn.calibration import calibration_curve
import xgboost as xgb
import lightgbm as lgb
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GRU, LSTM, Dropout, Conv1D, Input, Bidirectional, Layer
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import json
import warnings
import os
import time
import optuna
# from optuna.integration import TFKerasPruningCallback # No longer needed
import matplotlib.pyplot as plt
import math
import sys

# ---
# --- 1. GLOBAL SETTINGS & OFFLINE-TUNED HYPERPARAMETERS ---
# ---

# --- 1.a. Experiment Settings ---
N_RUNS = 5
GLOBAL_SEED = 42
N_TRIALS_LGBM = 25    # LGBM tuning (runs every time, but is fast)
N_TRIALS_STACKER = 25 # Stacker tuning (runs every time, but is fast)

# --- 1.b. Keras Loss Weights ---
# Justification: From offline 'loss weight analysis.png'.
# Best performance (lowest val_rmse_mm: 3.4628) was found with these weights.
HP_LOSS_WEIGHTS = {'prob_output': 0.2, 'reg_output': 1.8}

# --- 1.c. TCN-BiGRU Hyperparameters ---
# Justification: From offline Optuna tuning script.
# Best performing trial (Val Loss: 0.3507)
HP_TCN = {
    'conv_filters': 128, 
    'gru_units': 96, 
    'dropout_rate': 0.384665661176, 
    'learning_rate': 0.0009239150319627245, 
    'l2_reg': 0.0004138040112561013
}

# --- 1.d. BiLSTM Hyperparameters ---
# Justification: From offline Optuna tuning script.
# Best performing trial (Val Loss: 0.1894)
HP_LSTM = {
    'lstm_units_1': 192, 
    'lstm_units_2': 192, 
    'dropout_rate': 0.18499024021501972, 
    'learning_rate': 0.000551650239861584, 
    'l2_reg': 5.0381193632061425e-05
}

# --- 1.e. Environment Setup ---
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['PYTHONHASHSEED'] = str(GLOBAL_SEED)

# --- 1.1. REPRODUCIBILITY LOG ---
def get_pkg_version(package_name):
    try:
        return importlib.metadata.version(package_name)
    except importlib.metadata.PackageNotFoundError:
        try:
            return pkg_resources.get_distribution(package_name).version
        except:
            return "Not Found"

def print_reproducibility_info():
    """Prints a full reproducibility log."""
    print("="*50)
    print("     REPRODUCIBILITY LOG ")
    print("="*50)
    print(f"  Python Version: {sys.version.split(' ')[0]}")
    packages = ['pandas', 'numpy', 'scikit-learn', 'tensorflow', 
                'lightgbm', 'xgboost', 'optuna', 'scipy']
    for pkg in packages:
        print(f"  {pkg} Version: {get_pkg_version(pkg)}")
    
    try:
        gpus = tf.config.experimental.list_physical_devices('GPU')
        if gpus:
            for i, gpu in enumerate(gpus):
                details = tf.config.experimental.get_device_details(gpu)
                print(f"  GPU {i}: {details.get('name', 'N/A')}") # Use .get for safety
        else:
            print("  GPU: None Found. Using CPU.")
    except Exception as e:
        print(f"  Could not get GPU info: {e}")
    print("="*50 + "\n")

print_reproducibility_info()
print(f"Num GPUs Available: {len(tf.config.experimental.list_physical_devices('GPU'))}")

# --- 2. SETUP MULTI-GPU STRATEGY ---
try:
    strategy = tf.distribute.MirroredStrategy()
    print(f' Multi-GPU strategy active. Number of devices: {strategy.num_replicas_in_sync}')
    GLOBAL_BATCH_SIZE = 64 * strategy.num_replicas_in_sync
    print(f'Global Batch Size: {GLOBAL_BATCH_SIZE}')
except Exception as e:
    print(f" Could not initialize MirroredStrategy: {e}. Falling back to default strategy.")
    strategy = tf.distribute.get_strategy()
    GLOBAL_BATCH_SIZE = 64

# --- 3. CUSTOM LOSS & ATTENTION LAYER ---
class Attention(Layer):
    def __init__(self,**kwargs): super(Attention,self).__init__(**kwargs)
    def build(self, input_shape):
        self.W=self.add_weight(name='att_W',shape=(input_shape[-1],1),initializer='glorot_uniform',trainable=True)
        self.b=self.add_weight(name='att_b',shape=(input_shape[1],1),initializer='zeros',trainable=True); super(Attention,self).build(input_shape)
    def call(self, x): e=tf.tanh(tf.tensordot(x,self.W,axes=1)+self.b); a=tf.nn.softmax(e,axis=1); o=x*a; return tf.reduce_sum(o,axis=1)

# ---
# --- 4. MODEL DEFINITIONS (Using Tuned Hyperparameters) ---
# ---
def create_tcn_bigru_model(n_timesteps, n_features):
    """Creates a TCN-BiGRU model with OFFLINE-TUNED hyperparameters."""
    reg = l2(HP_TCN['l2_reg'])
    i=Input(shape=(n_timesteps,n_features))
    
    t=Conv1D(HP_TCN['conv_filters'], 3, activation='relu', padding='causal', dilation_rate=1, kernel_regularizer=reg)(i)
    t=Dropout(HP_TCN['dropout_rate'])(t)
    t=Conv1D(HP_TCN['conv_filters'], 3, activation='relu', padding='causal', dilation_rate=2, kernel_regularizer=reg)(t)
    t=Dropout(HP_TCN['dropout_rate'])(t)
    t=Conv1D(HP_TCN['conv_filters'], 3, activation='relu', padding='causal', dilation_rate=4, kernel_regularizer=reg)(t)
    t=Dropout(HP_TCN['dropout_rate'])(t)
    
    bg=Bidirectional(GRU(HP_TCN['gru_units'], activation='tanh', return_sequences=True, kernel_regularizer=reg))(t)
    att=Attention()(bg)
    p=Dense(1,activation='sigmoid',name='prob_output')(att)
    r=Dense(1,activation='linear',name='reg_output',dtype='float32')(att)
    return Model(inputs=i,outputs=[p,r])

def create_bilstm_model(n_timesteps, n_features):
    """Creates a BiLSTM model with OFFLINE-TUNED hyperparameters."""
    reg = l2(HP_LSTM['l2_reg'])
    i=Input(shape=(n_timesteps,n_features))
    
    l=Bidirectional(LSTM(HP_LSTM['lstm_units_1'], activation='tanh', return_sequences=True, kernel_regularizer=reg))(i)
    l=Dropout(HP_LSTM['dropout_rate'])(l)
    l=Bidirectional(LSTM(HP_LSTM['lstm_units_2'], activation='tanh', return_sequences=True, kernel_regularizer=reg))(l)
    l=Dropout(HP_LSTM['dropout_rate'])(l)
    
    att=Attention()(l)
    p=Dense(1,activation='sigmoid',name='prob_output')(att)
    r=Dense(1,activation='linear',name='reg_output',dtype='float32')(att)
    return Model(inputs=i,outputs=[p,r])

def compile_keras_model(model, model_type):
    """Compiles a Keras model with OFFLINE-TUNED learning rate and loss weights."""
    if model_type == 'tcn':
        lr = HP_TCN['learning_rate']
    elif model_type == 'lstm':
        lr = HP_LSTM['learning_rate']
    else:
        # Fallback, though should not be reached
        lr = 0.0002 
        
    model.compile(optimizer=Adam(learning_rate=lr),
                  loss={'prob_output':BinaryCrossentropy(), 'reg_output':'mean_squared_error'},
                  loss_weights=HP_LOSS_WEIGHTS) # Use global weights
    return model

def apply_soft_gate(y_p,y_r):
    ypf=y_p.astype(np.float64).flatten(); yrf=y_r.astype(np.float64).flatten(); ypf=np.nan_to_num(ypf); yrf=np.nan_to_num(yrf); return yrf*ypf

# --- 5. DATA PREPARATION (GLOBAL) ---
DATASET_NAME = "imphal"
try:
    df=pd.read_csv(f'{DATASET_NAME}.csv')
    print(f" Successfully loaded '{DATASET_NAME}.csv'")
except FileNotFoundError:
    print(f" Error: '{DATASET_NAME}.csv' not found. Trying fallback...")
    try:
        df=pd.read_csv(f'/kaggle/input/dataset2/{DATASET_NAME}.csv')
        print(f" Successfully loaded from fallback path.")
    except FileNotFoundError:
        print(f" Error: Fallback path also failed. Exiting.")
        exit()

# Pre-processing
df['Date']=pd.to_datetime({'year':df['YEAR'],'month':df['MN'],'day':df['DT']}); df.set_index('Date',inplace=True)
num_cols=['RF','MSLP','DBT','RH','ONI','DMI']; [df[c].apply(pd.to_numeric,errors='coerce') for c in num_cols]
daily=df.groupby(df.index.date).agg({c:('sum' if c=='RF' else 'mean') for c in num_cols}); daily.index=pd.to_datetime(daily.index); daily.dropna(subset=num_cols,inplace=True)
daily['RF_SSA']=daily['RF'].rolling(7,center=True).mean().fillna(daily['RF']); daily['RF_7m']=daily['RF'].rolling(7).mean().shift(1); daily['RF_7s']=daily['RF'].rolling(7).std().shift(1); daily['sin_d']=np.sin(2*np.pi*daily.index.dayofyear/365.25); daily['cos_d']=np.cos(2*np.pi*daily.index.dayofyear/365.25)
daily['RF_l1']=daily['RF'].shift(1); daily['DBT_l1']=daily['DBT'].shift(1); daily['RH_MSLP_I']=(daily['RH']/daily['MSLP']).replace([np.inf,-np.inf],np.nan); daily.replace([np.inf,-np.inf],np.nan,inplace=True); daily.dropna(inplace=True); daily['RF']=pd.to_numeric(daily['RF'],errors='coerce').fillna(0)
HORIZON=3; LOOK_BACK=7; FEATURES=['RF_SSA','MSLP','DBT','RH','RF_7m','RF_7s','sin_d','cos_d','ONI','DMI','RF_l1','DBT_l1','RH_MSLP_I']; FEATURES=[f for f in FEATURES if f in daily.columns]; N_FEATS=len(FEATURES); print(f"Features ({N_FEATS}): {FEATURES}")
RF_I=FEATURES.index('RF_SSA') if 'RF_SSA' in FEATURES else -1; MSLP_I=FEATURES.index('MSLP') if 'MSLP' in FEATURES else -1; DBT_I=FEATURES.index('DBT') if 'DBT' in FEATURES else -1; RH_I=FEATURES.index('RH') if 'RH' in FEATURES else -1; ONI_I=FEATURES.index('ONI') if 'ONI' in FEATURES else -1; DMI_I=FEATURES.index('DMI') if 'DMI' in FEATURES else -1; SIN_I=FEATURES.index('sin_d') if 'sin_d' in FEATURES else -1; COS_I=FEATURES.index('cos_d') if 'cos_d' in FEATURES else -1; RF_L1_I=FEATURES.index('RF_l1') if 'RF_l1' in FEATURES else -1; DBT_L1_I=FEATURES.index('DBT_l1') if 'DBT_l1' in FEATURES else -1

def create_ts_feats(data,feats,target,lb,hz):
    X,Y=[],[]; ts=data[target].shift(-hz); ts=pd.to_numeric(ts,errors='coerce'); vfd=data[feats].dropna(); max_idx=len(vfd)-lb-hz+1; [ (X.append(fw),Y.append(ts.iloc[i+lb-1])) for i in range(max_idx) if (i+lb-1)<len(ts) and not pd.isna(ts.iloc[i+lb-1]) and (fw:=vfd.iloc[i:i+lb].values).shape[0]==lb ]; X=np.array(X).astype(np.float32); Y=np.array(Y).astype(np.float32); print(f"X:{X.shape}, Y:{Y.shape}"); return X,Y

X_seq_o, Y_o = create_ts_feats(daily, FEATURES, 'RF', LOOK_BACK, HORIZON); v_idx=~np.isnan(Y_o); X_seq_o=X_seq_o[v_idx]; Y_o=Y_o[v_idx]
Y_o = np.log1p(Y_o)
print(f"Total usable samples: {X_seq_o.shape[0]}")

# --- 6. LGBM FEATURE ENGINEERING (GLOBAL) ---
def get_slope(y):
    y = pd.to_numeric(y, errors='coerce').astype(np.float64)
    if np.isnan(y).all(): return 0.0
    y = np.nan_to_num(y); x = np.arange(len(y), dtype=np.float64); variance_x = np.var(x); covariance_xy = np.mean(x * y) - np.mean(x) * np.mean(y)
    if variance_x == 0: return 0.0
    return covariance_xy / variance_x

def create_lgbm_feats(seq_data_orig):
    seq_data_float = seq_data_orig.astype(np.float64)
    def safe_slice(data, idx):
        if 0 <= idx < data.shape[2]: return data[:, :, idx]
        else: return np.zeros((data.shape[0], data.shape[1]))
    rf,dbt,rh,mslp=safe_slice(seq_data_float,RF_I),safe_slice(seq_data_float,DBT_I),safe_slice(seq_data_float,RH_I),safe_slice(seq_data_float,MSLP_I)
    ld={'rf_l':rf[:,-1],'dbt_l':dbt[:,-1],'rh_l':rh[:,-1],'mslp_l':mslp[:,-1],'rl1_l':safe_slice(seq_data_float,RF_L1_I)[:,-1],'dl1_l':safe_slice(seq_data_float,DBT_L1_I)[:,-1],'oni_l':safe_slice(seq_data_float,ONI_I)[:,-1],'dmi_l':safe_slice(seq_data_float,DMI_I)[:,-1],'sin_l':safe_slice(seq_data_float,SIN_I)[:,-1],'cos_l':safe_slice(seq_data_float,COS_I)[:,-1]}
    st={'rf_m':np.nanmean(rf,1),'rf_s':np.nanstd(rf,1),'rf_max':np.nanmax(rf,1),'rf_min':np.nanmin(rf,1),'rf_sum':np.nansum(rf,1),'rf_q25':np.nanquantile(rf,0.25,1),'rf_q75':np.nanquantile(rf,0.75,1),
        'dbt_m':np.nanmean(dbt,1),'dbt_s':np.nanstd(dbt,1),'dbt_max':np.nanmax(dbt,1),'dbt_min':np.nanmin(dbt,1),'dbt_q25':np.nanquantile(dbt,0.25,1),'dbt_q75':np.nanquantile(dbt,0.75,1),
        'rh_m':np.nanmean(rh,1),'rh_s':np.nanstd(rh,1),'rh_max':np.nanmax(rh,1),'rh_min':np.nanmin(rh,1),'rh_q25':np.nanquantile(rh,0.25,1),'rh_q75':np.nanquantile(rh,0.75,1),
        'mslp_m':np.nanmean(mslp,1),'mslp_s':np.nanstd(mslp,1),'mslp_max':np.nanmax(mslp,1),'mslp_min':np.nanmin(mslp,1),'mslp_q25':np.nanquantile(mslp,0.25,1),'mslp_q75':np.nanquantile(mslp,0.75,1)}
    tr={'rf_sl':np.apply_along_axis(get_slope,1,rf),'dbt_sl':np.apply_along_axis(get_slope,1,dbt),'rh_sl':np.apply_along_axis(get_slope,1,rh),'mslp_sl':np.apply_along_axis(get_slope,1,mslp),'rd':np.nansum(rf>.1,1),'hrd':np.nansum(rf>20.0,1),'rf_d':rf[:,-1]-rf[:,-2] if rf.shape[1]>1 else 0,'dbt_d':dbt[:,-1]-dbt[:,-2] if dbt.shape[1]>1 else 0,'rh_d':rh[:,-1]-rh[:,-2] if rh.shape[1]>1 else 0}
    f_d={**ld,**st,**tr}; return pd.DataFrame(f_d).fillna(0).values


def mean_absolute_percentage_error_safe(y_true, y_pred, eps=1e-6):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = y_true > eps
    if np.sum(mask) == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def nash_sutcliffe_efficiency(y_true, y_pred):
    denom = np.sum((y_true - np.mean(y_true)) ** 2)
    if denom == 0:
        return np.nan
    return 1 - np.sum((y_true - y_pred) ** 2) / denom


def bias_metric(y_true, y_pred):
    return np.mean(y_pred - y_true)


# --- 7. HELPER FUNCTIONS FOR METRICS, PLOTS, ETC. ---
def calc_stratified_metrics(y_t, y_p, name, is_main_model=False):
    """
    FULL hydrological metrics for IMPHAL
    Metrics : R2, MAE, MSE, RMSE, MAPE, Bias, NSE, Samples
    Regimes : Overall | No Rain | Moderate | Heavy
    """

    y_t_f = np.nan_to_num(np.expm1(y_t))
    y_p = np.nan_to_num(np.maximum(0, y_p))

    bins = {
        'Overall': (y_t_f >= 0),
        'No Rain (0mm)': (y_t_f == 0),
        'Mod (0-10mm)': (y_t_f > 0) & (y_t_f <= 10),
        'Heavy (>10mm)': (y_t_f > 10)
    }

    results = {
        'R2': {}, 'MAE': {}, 'MSE': {}, 'RMSE': {},
        'MAPE': {}, 'Bias': {}, 'NSE': {}, 'Samples': {}
    }

    for bin_name, idx in bins.items():
        yt, yp = y_t_f[idx], y_p[idx]

        if len(yt) == 0:
            for k in results:
                results[k][bin_name] = np.nan if k != 'Samples' else 0
        else:
            mse = mean_squared_error(yt, yp)
            results['R2'][bin_name] = np.nan if np.var(yt) < 1e-9 else r2_score(yt, yp)
            results['MAE'][bin_name] = mean_absolute_error(yt, yp)
            results['MSE'][bin_name] = mse
            results['RMSE'][bin_name] = np.sqrt(mse)
            results['MAPE'][bin_name] = mean_absolute_percentage_error_safe(yt, yp)
            results['Bias'][bin_name] = bias_metric(yt, yp)
            results['NSE'][bin_name] = nash_sutcliffe_efficiency(yt, yp)
            results['Samples'][bin_name] = len(yt)

    # Optional print for main ensemble (first run only)
    if is_main_model:
        print(f"\n--- {name} (FULL HYDROLOGICAL METRICS) ---")
        for metric in ['R2', 'MAE', 'RMSE', 'MAPE', 'Bias', 'NSE']:
            print(metric, {b: f"{results[metric][b]:.4f}" for b in bins})

    return {
        f"{name}_{bin_name}_{metric}": val
        for metric, bin_vals in results.items()
        for bin_name, val in bin_vals.items()
    }


def get_model_complexity(model, model_type):
    params = 0
    if model_type == 'keras': params = model.count_params()
    elif model_type == 'lgbm': params = model.n_features_in_ * model.n_estimators_
    elif model_type == 'xgb': params = -1 
    return {"params": params}

def get_inference_latency(model, data, model_type, batch_size=GLOBAL_BATCH_SIZE):
    try:
        start_time = time.time()
        if model_type == 'keras': model.predict(data, batch_size=batch_size, verbose=0)
        elif model_type in ['lgbm', 'xgb']: model.predict(data)
        end_time = time.time()
        return {"latency_ms_per_sample": (end_time - start_time) * 1000 / len(data)}
    except Exception as e:
        print(f"Error calculating latency: {e}")
        return {"latency_ms_per_sample": -1}

def print_run_summary(run_metrics, run_complexity, run_seed):
    print("\n" + "="*50)
    print(f"     PERFORMANCE SUMMARY FOR RUN {run_seed} ")
    print("="*50 + "\n")
    metrics_s = pd.Series(run_metrics)
    print("--- 🚀 Key Metric Summary (MAIN ENSEMBLE) ---")
    main_metrics = [
        'MAIN_ENSEMBLE_Overall_R2', 'MAIN_ENSEMBLE_Overall_MAE', 'MAIN_ENSEMBLE_Overall_RMSE',
        'MAIN_ENSEMBLE_Mod (0-10mm)_MAE', 'MAIN_ENSEMBLE_Mod (0-10mm)_R2',
        'MAIN_ENSEMBLE_Heavy (>10mm)_MAE', 'MAIN_ENSEMBLE_Heavy (>10mm)_RMSE'
    ]
    key_results = metrics_s[metrics_s.index.isin(main_metrics)]
    if not key_results.empty:
        for idx, val in key_results.items(): print(f"  {idx:<30}: {val:.4f}")
    else: print("  No main metrics to display.")
    print("\n---  Ablation: Gating (Overall_MAE) ---")
    ablation_metrics = [
        'ABL_TCN_Gated_Overall_MAE', 'ABL_TCN_Direct_Overall_MAE',
        'ABL_LSTM_Gated_Overall_MAE', 'ABL_LSTM_Direct_Overall_MAE',
        'ABL_LGBM_Gated_Overall_MAE', 'ABL_LGBM_Direct_Overall_MAE'
    ]
    ablation_results = metrics_s[metrics_s.index.isin(ablation_metrics)]
    if not ablation_results.empty:
        for idx, val in ablation_results.items(): print(f"  {idx:<30}: {val:.4f}")
    else: print("  No ablation metrics to display.")
    print("\n--- 📉 Baselines (Overall_MAE) ---")
    baseline_metrics = [
        'BASELINE_Persistence_Overall_MAE', 'BASELINE_Persistence_Overall_RMSE',
        'BASELINE_Climatology_Overall_MAE', 'BASELINE_Climatology_Overall_RMSE'
    ]
    baseline_results = metrics_s[metrics_s.index.isin(baseline_metrics)]
    if not baseline_results.empty:
        for idx, val in baseline_results.items(): print(f"  {idx:<30}: {val:.4f}")
    else: print("  No baseline metrics to display.")
    print("\n---  Complexity & Latency ---")
    if run_complexity:
        print(pd.DataFrame(run_complexity).T.to_markdown(floatfmt=".2f"))
    else: print("  No complexity data to display.")

def plot_calibration_curves(y_true_prob, prob_preds, model_names, dataset_name, run_seed):
    print("\n---  Generating Calibration Plots (First Run Only) ---")
    plt.figure(figsize=(10, 8))
    ax = plt.subplot(111)
    ax.plot([0, 1], [0, 1], "k:", label="Perfectly calibrated")
    for i, (y_prob, name) in enumerate(zip(prob_preds, model_names)):
        try:
            y_prob = np.nan_to_num(np.clip(y_prob, 0, 1))
            fraction_of_positives, mean_predicted_value = calibration_curve(
                y_true_prob, y_prob, n_bins=10, strategy='uniform'
            )
            ax.plot(mean_predicted_value, fraction_of_positives, "s-", 
                    label=f"{name}", markersize=8)
        except Exception as e:
            print(f"   Could not plot calibration for {name}: {e}")
    ax.set_title("Calibration Plots (Probability of Rain)")
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Fraction of positives (True rain)")
    ax.set_ylim([-0.05, 1.05])
    ax.legend(loc="lower right")
    ax.grid(True, linestyle='--', alpha=0.6)
    plot_fn = f"{dataset_name}_calibration_plots_Run{run_seed}.png"
    plt.tight_layout()
    plt.savefig(plot_fn, dpi=150)
    print(f"---  Calibration plot saved to '{plot_fn}' ---")
    plt.close()

# --- 8. MAIN EXPERIMENT FUNCTION ---
def run_experiment(run_seed):
    print(f"\n{'='*20} STARTING RUN {run_seed} {'='*20}")
    
    # --- 8.1. SET SEEDS ---
    np.random.seed(run_seed)
    tf.keras.utils.set_random_seed(run_seed)
    run_metrics = {}
    run_complexity = {}

    # --- 8.2. CREATE DATA SPLITS (Solves P5) ---
    sp_idx=int(X_seq_o.shape[0]*.8)
    if sp_idx < 1 or sp_idx >= X_seq_o.shape[0] - 1:
        print(f" Invalid split index: {sp_idx}. Not enough data.")
        return {}, {}
        
    X_tr_v_o, X_ts_o = X_seq_o[:sp_idx], X_seq_o[sp_idx:]
    Y_tr_v_r, Y_ts_r = Y_o[:sp_idx], Y_o[sp_idx:]
    Y_tr_v_p, Y_ts_p = (np.expm1(Y_tr_v_r)>0).astype(int), (np.expm1(Y_ts_r)>0).astype(int)

    sc=StandardScaler()
    X_tr_v_f_s=sc.fit_transform(np.nan_to_num(X_tr_v_o.reshape(X_tr_v_o.shape[0],-1)))
    X_tr_v_s=X_tr_v_f_s.reshape(X_tr_v_o.shape).astype(np.float32)
    X_ts_f_s=sc.transform(np.nan_to_num(X_ts_o.reshape(X_ts_o.shape[0],-1)))
    X_ts_s=X_ts_f_s.reshape(X_ts_o.shape).astype(np.float32)

    X_tr_v_nf = create_lgbm_feats(X_tr_v_o)
    X_ts_nf = create_lgbm_feats(X_ts_o)
    s_nf = StandardScaler()
    X_tr_v_nf_s = s_nf.fit_transform(np.nan_to_num(X_tr_v_nf))
    X_ts_nf_s = s_nf.transform(np.nan_to_num(X_ts_nf))

    (X_tr_s, X_v_s,
     X_tr_s_o_s, X_v_s_o_s,
     X_tr_nf_s, X_v_nf_s, 
     Y_tr_r, Y_v_r,
     Y_tr_p, Y_v_p) = train_test_split(
        X_tr_v_s, X_tr_v_o, X_tr_v_nf_s, Y_tr_v_r, Y_tr_v_p, 
        test_size=0.2, random_state=run_seed
    )
    
    X_tr_fc = np.hstack([X_tr_s.reshape(X_tr_s.shape[0], -1), X_tr_nf_s])
    X_v_fc = np.hstack([X_v_s.reshape(X_v_s.shape[0], -1), X_v_nf_s])
    X_ts_fc = np.hstack([X_ts_s.reshape(X_ts_s.shape[0], -1), X_ts_nf_s])
    
    print("Data shapes for this run:")
    print(f"  Train (Seq): {X_tr_s.shape} | Val (Seq): {X_v_s.shape} | Test (Seq): {X_ts_s.shape}")
    print(f"  Train (LGBM): {X_tr_fc.shape} | Val (LGBM): {X_v_fc.shape} | Test (LGBM): {X_ts_fc.shape}")

    # ---
    # --- 8.3. TUNE L0 LGBM MODELS --- (Keras Tuning Removed)
    # ---
    print("\n [GPU] LGBM Tuning...")
    Y_tr_r_l=np.nan_to_num(Y_tr_r); Y_v_r_l=np.nan_to_num(Y_v_r)
    
    def obj_r_lgbm(t):
        p={'objective':'regression_l1','metric':'rmse','n_estimators':t.suggest_int('n_estimators',400,1500),
           'learning_rate':t.suggest_float('learning_rate',.01,.1, log=True),
           'num_leaves':t.suggest_int('num_leaves',20,80),
           'max_depth':t.suggest_int('max_depth',5,15),
           'reg_alpha':t.suggest_float('reg_alpha',0,1), 'reg_lambda':t.suggest_float('reg_lambda',0,1),
           'colsample_bytree':t.suggest_float('colsample_bytree',.6,1),
           'subsample':t.suggest_float('subsample',.6,1),
           'n_jobs':-1,'random_state':run_seed,'verbose':-1, 'device': 'gpu'}
        m=lgb.LGBMRegressor(**p); m.fit(X_tr_fc, Y_tr_r_l, eval_set=[(X_v_fc, Y_v_r_l)], eval_metric='rmse', callbacks=[lgb.early_stopping(15, verbose=False)])
        pr=m.predict(X_v_fc); return np.sqrt(mean_squared_error(Y_v_r_l, pr))

    def obj_c_lgbm(t):
        p={'objective':'binary','metric':'logloss','n_estimators':t.suggest_int('n_estimators',400,1500),
           'learning_rate':t.suggest_float('learning_rate',.01,.1, log=True),
           'num_leaves':t.suggest_int('num_leaves',20,80),
           'max_depth':t.suggest_int('max_depth',5,15),
           'reg_alpha':t.suggest_float('reg_alpha',0,1), 'reg_lambda':t.suggest_float('reg_lambda',0,1),
           'n_jobs':-1,'random_state':run_seed,'verbose':-1, 'device': 'gpu'}
        m=lgb.LGBMClassifier(**p); m.fit(X_tr_fc, Y_tr_p, eval_set=[(X_v_fc, Y_v_p)], eval_metric='logloss', callbacks=[lgb.early_stopping(15, verbose=False)])
        pr=m.predict_proba(X_v_fc)[:, 1]; return log_loss(Y_v_p, pr)

    s_r=optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=run_seed))
    s_r.optimize(obj_r_lgbm, n_trials=N_TRIALS_LGBM, show_progress_bar=(run_seed==GLOBAL_SEED))
    s_c=optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=run_seed))
    s_c.optimize(obj_c_lgbm, n_trials=N_TRIALS_LGBM, show_progress_bar=(run_seed==GLOBAL_SEED))
    
    bp_r=s_r.best_params; bp_c=s_c.best_params
    bp_r.update({'objective':'regression_l1','metric':'rmse','n_jobs':-1,'random_state':run_seed,'verbose':-1, 'device': 'gpu'})
    bp_c.update({'objective':'binary','metric':'logloss','n_jobs':-1,'random_state':run_seed,'verbose':-1, 'device': 'gpu'})
    print(" [GPU] LGBM Tuned.")

    # --- 8.4. K-FOLD OOF GENERATION ---
    N_SPLITS = 5
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=run_seed)
    X_oof_s = X_tr_v_s
    X_oof_fc = np.hstack([X_tr_v_s.reshape(X_tr_v_s.shape[0], -1), X_tr_v_nf_s])
    Y_oof_r = Y_tr_v_r
    Y_oof_p = Y_tr_v_p
    
    oof_p_tcn_p = np.zeros(len(Y_oof_r)); oof_p_tcn_r = np.zeros(len(Y_oof_r))
    oof_p_lstm_p = np.zeros(len(Y_oof_r)); oof_p_lstm_r = np.zeros(len(Y_oof_r))
    oof_p_lgbm_p = np.zeros(len(Y_oof_r)); oof_p_lgbm_r = np.zeros(len(Y_oof_r))
    
    es_kfold = EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True)

    print(f"\n--- Starting K-Fold OOF Generation ({N_SPLITS} Folds) ---")
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_oof_s, Y_oof_r)):
        if run_seed == GLOBAL_SEED: print(f"--- FOLD {fold+1}/{N_SPLITS} ---")
        
        X_tr_fold_s, X_val_fold_s = X_oof_s[train_idx], X_oof_s[val_idx]
        X_tr_fold_fc, X_val_fold_fc = X_oof_fc[train_idx], X_oof_fc[val_idx]
        Y_tr_fold_r, Y_val_fold_r = Y_oof_r[train_idx], Y_oof_r[val_idx]
        Y_tr_fold_p, Y_val_fold_p = Y_oof_p[train_idx], Y_oof_p[val_idx]
        Y_tr_fold_targets = {'prob_output': Y_tr_fold_p, 'reg_output': Y_tr_fold_r}
        Y_val_fold_targets = {'prob_output': Y_val_fold_p, 'reg_output': Y_val_fold_r}

        # --- 1. TCN-BiGRU (K-Fold) ---
        tf.keras.backend.clear_session()
        with strategy.scope():
            m_tcn = create_tcn_bigru_model(LOOK_BACK, N_FEATS)
            m_tcn = compile_keras_model(m_tcn, 'tcn') # Pass model type
        m_tcn.fit(X_tr_fold_s, Y_tr_fold_targets,
                  validation_data=(X_val_fold_s, Y_val_fold_targets),
                  epochs=120, callbacks=[es_kfold],
                  batch_size=GLOBAL_BATCH_SIZE, verbose=0)
        p, r = m_tcn.predict(X_val_fold_s, batch_size=GLOBAL_BATCH_SIZE, verbose=0); oof_p_tcn_p[val_idx]=p.flatten(); oof_p_tcn_r[val_idx]=r.flatten()

        # --- 2. BiLSTM (K-Fold) ---
        tf.keras.backend.clear_session()
        with strategy.scope():
            m_lstm = create_bilstm_model(LOOK_BACK, N_FEATS)
            m_lstm = compile_keras_model(m_lstm, 'lstm') # Pass model type
        m_lstm.fit(X_tr_fold_s, Y_tr_fold_targets,
                   validation_data=(X_val_fold_s, Y_val_fold_targets),
                   epochs=120, callbacks=[es_kfold],
                   batch_size=GLOBAL_BATCH_SIZE, verbose=0)
        p, r = m_lstm.predict(X_val_fold_s, batch_size=GLOBAL_BATCH_SIZE, verbose=0); oof_p_lstm_p[val_idx]=p.flatten(); oof_p_lstm_r[val_idx]=r.flatten()

        # --- 3. LGBM (K-Fold) ---
        m_r_fold = lgb.LGBMRegressor(**bp_r); m_r_fold.fit(X_tr_fold_fc, Y_tr_fold_r); oof_p_lgbm_r[val_idx] = m_r_fold.predict(X_val_fold_fc)
        m_p_fold = lgb.LGBMClassifier(**bp_c); m_p_fold.fit(X_tr_fold_fc, Y_tr_fold_p); oof_p_lgbm_p[val_idx] = m_p_fold.predict_proba(X_val_fold_fc)[:, 1]

    print("---  K-Fold OOF Generation Complete ---")

    # --- 8.5. TUNE & TRAIN L1 STACKER (XGBoost) ---
    print("\n--- Creating Enriched OOF Meta-Dataset ---")
    oof_p_tcn_p, oof_p_tcn_r = np.nan_to_num(oof_p_tcn_p), np.nan_to_num(oof_p_tcn_r)
    oof_p_lstm_p, oof_p_lstm_r = np.nan_to_num(oof_p_lstm_p), np.nan_to_num(oof_p_lstm_r)
    oof_p_lgbm_p, oof_p_lgbm_r = np.nan_to_num(oof_p_lgbm_p), np.nan_to_num(oof_p_lgbm_r)

    y_p_v_tcn_log = np.maximum(0, apply_soft_gate(oof_p_tcn_p, oof_p_tcn_r))
    y_p_v_lstm_log = np.maximum(0, apply_soft_gate(oof_p_lstm_p, oof_p_lstm_r))
    y_p_v_lgbm_log = np.maximum(0, apply_soft_gate(oof_p_lgbm_p, oof_p_lgbm_r))
    oof_p_tcn_r_log = np.maximum(0, oof_p_tcn_r)
    oof_p_lstm_r_log = np.maximum(0, oof_p_lstm_r)
    oof_p_lgbm_r_log = np.maximum(0, oof_p_lgbm_r)

    X_meta_tr = np.stack([
        oof_p_tcn_p, oof_p_tcn_r_log, y_p_v_tcn_log,
        oof_p_lstm_p, oof_p_lstm_r_log, y_p_v_lstm_log,
        oof_p_lgbm_p, oof_p_lgbm_r_log, y_p_v_lgbm_log
    ], axis=1)
    
    X_meta_tr_enriched = np.hstack([X_meta_tr, X_oof_fc])
    Y_meta_tr = np.nan_to_num(Y_oof_r)

    print(f"Enriched meta-features shape: {X_meta_tr_enriched.shape}")
    print("\n--- Tuning XGBoost Stacker on OOF Data ---")
    
    def objective_xgb_stacker(trial):
        params = {'objective':'reg:squarederror', 'eval_metric':'rmse',
                  'n_estimators':trial.suggest_int('n_estimators',100,600),
                  'learning_rate':trial.suggest_float('learning_rate',0.01,0.2,log=True),
                  'max_depth':trial.suggest_int('max_depth',3,8),
                  'subsample':trial.suggest_float('subsample',0.6,1.0),
                  'colsample_bytree':trial.suggest_float('colsample_bytree',0.6,1.0),
                  'reg_alpha':trial.suggest_float('reg_alpha',0.0,1.0),
                  'reg_lambda':trial.suggest_float('reg_lambda',0.0,1.0),
                  'random_state':run_seed,'n_jobs':-1, 'tree_method': 'gpu_hist'}
        kf_stack = KFold(n_splits=4, shuffle=True, random_state=run_seed); scores = []
        for train_idx, val_idx in kf_stack.split(X_meta_tr_enriched):
            X_tr_cv, X_v_cv = X_meta_tr_enriched[train_idx], X_meta_tr_enriched[val_idx]
            Y_tr_cv, Y_v_cv = Y_meta_tr[train_idx], Y_meta_tr[val_idx]
            model = xgb.XGBRegressor(**params)
            model.fit(X_tr_cv, Y_tr_cv, eval_set=[(X_v_cv, Y_v_cv)], early_stopping_rounds=15, verbose=False)
            preds = model.predict(X_v_cv)
            rmse = np.sqrt(mean_squared_error(Y_v_cv, np.maximum(0, preds)))
            scores.append(rmse)
        return np.mean(scores)

    study_xgb = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=run_seed))
    study_xgb.optimize(objective_xgb_stacker, n_trials=N_TRIALS_STACKER, show_progress_bar=(run_seed==GLOBAL_SEED))
    best_xgb_params = study_xgb.best_params; 
    
    best_xgb_params.update({'objective':'reg:pseudohubererror', 
                            'random_state':run_seed, 'n_jobs':-1, 'tree_method': 'gpu_hist'})

    print("\n--- Training Final Stacker with Best Params on Full OOF Data ---")
    meta_model = xgb.XGBRegressor(**best_xgb_params)
    meta_model.fit(X_meta_tr_enriched, Y_meta_tr)
    print(f" Meta (XGBoost Tuned) OK.")

    # --- 8.6. FINAL L0 MODEL TRAINING ---
    print("\n--- Starting Final L0 Model Training (Sequential) on FULL 80% Data ---")
    X_tr_v_s_full = X_tr_v_s.astype(np.float32)
    Y_tr_v_r_full = Y_tr_v_r
    Y_tr_v_p_full = Y_tr_v_p
    Y_tr_v_targets_full = {'prob_output':Y_tr_v_p_full, 'reg_output':Y_tr_v_r_full}
    X_tr_v_fc_full = np.hstack([X_tr_v_s.reshape(X_tr_v_s.shape[0], -1), X_tr_v_nf_s])

    print(" [GPU] TCN-BiGRU (Final)...")
    tf.keras.backend.clear_session()
    with strategy.scope():
        # --- MODIFIED: Use hard-coded model ---
        m_tcn = create_tcn_bigru_model(LOOK_BACK, N_FEATS)
        m_tcn = compile_keras_model(m_tcn, 'tcn') # Pass model type
    m_tcn.fit(X_tr_v_s_full, Y_tr_v_targets_full, epochs=200, batch_size=GLOBAL_BATCH_SIZE, verbose=1)
    print(" [GPU] TCN-BiGRU OK.")

    print(" [GPU] BiLSTM (Final)...")
    tf.keras.backend.clear_session()
    with strategy.scope():
        # --- MODIFIED: Use hard-coded model ---
        m_lstm = create_bilstm_model(LOOK_BACK, N_FEATS)
        m_lstm = compile_keras_model(m_lstm, 'lstm') # Pass model type
    m_lstm.fit(X_tr_v_s_full, Y_tr_v_targets_full, epochs=200, batch_size=GLOBAL_BATCH_SIZE, verbose=1)
    print(" [GPU] BiLSTM OK.")

    print(" [GPU] LGBM (Final)...")
    m_lr = lgb.LGBMRegressor(**bp_r); m_lr.fit(X_tr_v_fc_full, Y_tr_v_r_full)
    m_lp = lgb.LGBMClassifier(**bp_c); m_lp.fit(X_tr_v_fc_full, Y_tr_v_p_full)
    print(" [GPU] LGBM OK.")
    print("\n All final L0 models trained on full 80% data.")

    # --- 8.7. FINAL PREDICTIONS ON TEST SET ---
    print("\n--- Generating Final Preds on TEST Set ---")
    P_ts_tcn_p, P_ts_tcn_r = m_tcn.predict(X_ts_s, batch_size=GLOBAL_BATCH_SIZE, verbose=0)
    P_ts_lstm_p, P_ts_lstm_r = m_lstm.predict(X_ts_s, batch_size=GLOBAL_BATCH_SIZE, verbose=0)
    P_ts_lgbm_p = m_lp.predict_proba(X_ts_fc)[:, 1]
    P_ts_lgbm_r = m_lr.predict(X_ts_fc)

    P_ts_tcn_p, P_ts_tcn_r=np.nan_to_num(P_ts_tcn_p.flatten()), np.nan_to_num(P_ts_tcn_r.flatten())
    P_ts_lstm_p, P_ts_lstm_r=np.nan_to_num(P_ts_lstm_p.flatten()), np.nan_to_num(P_ts_lstm_r.flatten())
    P_ts_lgbm_p, P_ts_lgbm_r=np.nan_to_num(P_ts_lgbm_p), np.nan_to_num(P_ts_lgbm_r)

    y_p_ts_tcn_log = np.maximum(0, apply_soft_gate(P_ts_tcn_p, P_ts_tcn_r))
    y_p_ts_lstm_log = np.maximum(0, apply_soft_gate(P_ts_lstm_p, P_ts_lstm_r))
    y_p_ts_lgbm_log = np.maximum(0, apply_soft_gate(P_ts_lgbm_p, P_ts_lgbm_r))
    P_ts_tcn_r_log = np.maximum(0, P_ts_tcn_r)
    P_ts_lstm_r_log = np.maximum(0, P_ts_lstm_r)
    P_ts_lgbm_r_log = np.maximum(0, P_ts_lgbm_r)
    
    X_meta_ts=np.stack([
        P_ts_tcn_p, P_ts_tcn_r_log, y_p_ts_tcn_log,
        P_ts_lstm_p, P_ts_lstm_r_log, y_p_ts_lstm_log,
        P_ts_lgbm_p, P_ts_lgbm_r_log, y_p_ts_lgbm_log
    ], axis=1)
    X_meta_ts_enriched = np.hstack([X_meta_ts, X_ts_fc]) 
    y_p_ens_log = meta_model.predict(X_meta_ts_enriched)
    
    y_p_ens = np.maximum(0, np.expm1(y_p_ens_log))
    y_p_ts_tcn = np.maximum(0, np.expm1(y_p_ts_tcn_log))
    y_p_ts_lstm = np.maximum(0, np.expm1(y_p_ts_lstm_log))
    y_p_ts_lgbm = np.maximum(0, np.expm1(y_p_ts_lgbm_log))
    P_ts_tcn_r_mm = np.maximum(0, np.expm1(P_ts_tcn_r_log))
    P_ts_lstm_r_mm = np.maximum(0, np.expm1(P_ts_lstm_r_log))
    P_ts_lgbm_r_mm = np.maximum(0, np.expm1(P_ts_lgbm_r_log))
    
    threshold=0.1; y_p_ens[y_p_ens < threshold] = 0.0
    Y_ts_r_fin=np.nan_to_num(Y_ts_r)

    # --- 8.8. FINAL EVALUATION ---
    print("\n---  FINAL EVALUATION (RUN {run_seed}) ---")
    y_p_persistence = X_ts_o[:, -1, RF_I] 
    run_metrics.update(calc_stratified_metrics(Y_ts_r_fin, y_p_persistence, "BASELINE_Persistence"))
    mean_log_pred = Y_tr_v_r.mean() 
    y_p_climatology = np.full_like(Y_ts_r_fin, np.expm1(mean_log_pred))
    run_metrics.update(calc_stratified_metrics(Y_ts_r_fin, y_p_climatology, "BASELINE_Climatology"))
    
    print("\n--- Ablation Study: Gated (P*R) vs. Direct (R) ---")
    run_metrics.update(calc_stratified_metrics(Y_ts_r_fin, y_p_ts_tcn, "ABL_TCN_Gated"))
    run_metrics.update(calc_stratified_metrics(Y_ts_r_fin, P_ts_tcn_r_mm, "ABL_TCN_Direct"))
    run_metrics.update(calc_stratified_metrics(Y_ts_r_fin, y_p_ts_lstm, "ABL_LSTM_Gated"))
    run_metrics.update(calc_stratified_metrics(Y_ts_r_fin, P_ts_lstm_r_mm, "ABL_LSTM_Direct"))
    run_metrics.update(calc_stratified_metrics(Y_ts_r_fin, y_p_ts_lgbm, "ABL_LGBM_Gated"))
    run_metrics.update(calc_stratified_metrics(Y_ts_r_fin, P_ts_lgbm_r_mm, "ABL_LGBM_Direct"))

    is_main = (run_seed == GLOBAL_SEED)
    run_metrics.update(calc_stratified_metrics(Y_ts_r_fin, y_p_ens, "MAIN_ENSEMBLE", is_main_model=is_main))
    
    # --- 8.9. COMPLEXITY ANALYSIS ---
    print("\n---  Complexity & Latency Analysis ---")
    run_complexity['TCN_BiGRU'] = {**get_model_complexity(m_tcn, 'keras'), 
                                   **get_inference_latency(m_tcn, X_ts_s, 'keras')}
    run_complexity['BiLSTM'] = {**get_model_complexity(m_lstm, 'keras'), 
                                **get_inference_latency(m_lstm, X_ts_s, 'keras')}
    run_complexity['LGBM'] = {**get_model_complexity(m_lr, 'lgbm'), 
                                **get_inference_latency(m_lr, X_ts_fc, 'lgbm')}
    run_complexity['STACKER_XGB'] = {**get_model_complexity(meta_model, 'xgb'), 
                                     **get_inference_latency(meta_model, X_meta_ts_enriched, 'xgb')}
    if is_main: print(pd.DataFrame(run_complexity).T.to_markdown(floatfmt=".2f"))

    
    # ---
    # --- 8.11. VISUALS & JSON (MODIFIED) ---
    # ---
    
    # --- Initialize samples outside the if-block ---
    samples = [] 
    
    # --- Generate Plots ONLY on the first run (to save time) ---
    if run_seed == GLOBAL_SEED:
        print("\n---  Generating Samples & Plots (First Run Only) ---")
        N_SAMPLES=30; DAYS=3
        Y_ts_r_fin_mm = np.expm1(Y_ts_r_fin)
        if len(Y_ts_r_fin_mm) >= DAYS:
            starts=np.arange(len(Y_ts_r_fin_mm)-DAYS+1)
            if len(starts) > 0:
                n_smpl=min(N_SAMPLES,len(starts))
                idxs=np.random.choice(starts,n_smpl,replace=False)
                for c, s_idx in enumerate(idxs):
                    block={f"grp_{c+1}":[]}
                    for i in range(DAYS):
                        idx=s_idx+i
                        if idx<len(Y_ts_r_fin_mm):
                            d={"d":i+1,"i":int(idx),"a":float(Y_ts_r_fin_mm[idx]),"pE":float(y_p_ens[idx])}
                            block[f"grp_{c+1}"].append(d)
                    samples.append(block)

        if len(samples) > 0:
            N_COLS=5; N_ROWS=math.ceil(len(samples)/N_COLS); fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=(N_COLS*4, N_ROWS*3), sharey=False, squeeze=False); axes=axes.flatten()
            for i in range(len(samples)):
                ax=axes[i]; d_list=list(samples[i].values())[0];
                if not d_list: continue
                s_idx=d_list[0]['i']; days=[d['d'] for d in d_list]; acts=[d['a'] for d in d_list]; ens_ps=[d['pE'] for d in d_list]
                if days and acts and ens_ps: ax.plot(days, acts, 'o-', label='Actual', c='b', lw=2, ms=6); ax.plot(days, ens_ps, 's--', label='Ensemble', c='r', lw=2, ms=5); ax.set_title(f"S{i+1}(Idx {s_idx})", fontsize=10); ax.set_xticks([1,2,3]); ax.set_xticklabels(['D1','D2','D3']); ax.grid(True, ls='--', alpha=.6); ax.set_ylabel("Rain(mm)")
            f_ax = next((ax for ax in axes[:len(samples)] if ax.has_data()), None)
            if f_ax: h,l=f_ax.get_legend_handles_labels(); fig.legend(h,l,loc='upper center', bbox_to_anchor=(.5,1.02), ncol=2, fontsize=12)
            for j in range(len(samples), len(axes)): axes[j].axis('off')
            plt.tight_layout(rect=[0,0,1,.96]);
            plot_fn = f"{DATASET_NAME}_forecast_samples.png"
            plt.savefig(plot_fn, dpi=150)
            print(f"---  Plot saved to '{plot_fn}' ---")
            plt.close()

        plot_calibration_curves(
            y_true_prob=Y_ts_p,
            prob_preds=[P_ts_tcn_p, P_ts_lstm_p, P_ts_lgbm_p],
            model_names=['TCN-BiGRU (p)', 'BiLSTM (p)', 'LGBM (p)'],
            dataset_name=DATASET_NAME,
            run_seed=run_seed
        )

    # --- Save JSON for EVERY run ---
    print(f"\n---  Saving results for Run {run_seed} to JSON ---")
    class NumpyEncoder(json.JSONEncoder):
        def default(self, obj):
            if isinstance(obj, (np.integer, np.int64)): return int(obj)
            elif isinstance(obj, (np.floating, np.float64)): return float(obj)
            elif isinstance(obj, np.ndarray): return obj.tolist()
            return super(NumpyEncoder, self).default(obj)

    json_output = {"run_seed": run_seed, "metrics": run_metrics, "complexity": run_complexity, "samples": samples}
    out_fn = f"{DATASET_NAME}_RESULTS_Run{run_seed}.json"
    with open(out_fn, 'w') as f:
        json.dump(json_output, f, indent=4, cls=NumpyEncoder)
        print(f"--- RESULTS SAVED to '{out_fn}' ---")
            
    print(f"\n{'='*20} COMPLETED RUN {run_seed} {'='*20}")
    
    print_run_summary(run_metrics, run_complexity, run_seed)
    
    return run_metrics, run_complexity

# --- 9. MAIN EXECUTION LOOP ---
all_run_metrics = []
all_run_complexity = []

for i in range(N_RUNS):
    seed = GLOBAL_SEED + i
    metrics, complexity = run_experiment(run_seed=seed)
    if metrics:
        all_run_metrics.append(metrics)
        all_run_complexity.append(complexity)

print("\n\n" + "="*50)
print("    ALL RUNS COMPLETE - FINAL AGGREGATED RESULTS    ")
print("="*50 + "\n")

# --- 10. FINAL AGGREGATED RESULTS ---
if all_run_metrics:
    metrics_df = pd.DataFrame(all_run_metrics)
    metrics_mean = metrics_df.mean()
    metrics_std = metrics_df.std()
    results_summary = pd.DataFrame({'Mean': metrics_mean, 'Std': metrics_std})

    print("---  Key Metric Summary (Mean ± Std) ---")
    main_metrics = [
        'MAIN_ENSEMBLE_Overall_R2', 'MAIN_ENSEMBLE_Overall_MAE', 'MAIN_ENSEMBLE_Overall_RMSE',
        'MAIN_ENSEMBLE_Mod (0-10mm)_R2',
        'MAIN_ENSEMBLE_Mod (0-10mm)_MAE',
        'MAIN_ENSEMBLE_Heavy (>10mm)_MAE', 'MAIN_ENSEMBLE_Heavy (>10mm)_RMSE'
    ]
    main_metrics_exist = [m for m in main_metrics if m in results_summary.index]
    if main_metrics_exist:
        key_results = results_summary.loc[main_metrics_exist]
        for idx, row in key_results.iterrows():
            print(f"  {idx:<30}: {row['Mean']:.4f} ± {row['Std']:.4f}")
    else: print("  No main metrics to display.")

    print("\n---  Ablation: Gating (Mean ± Std) ---")
    ablation_metrics = [
        'ABL_TCN_Gated_Overall_MAE', 'ABL_TCN_Direct_Overall_MAE',
        'ABL_LSTM_Gated_Overall_MAE', 'ABL_LSTM_Direct_Overall_MAE',
        'ABL_LGBM_Gated_Overall_MAE', 'ABL_LGBM_Direct_Overall_MAE'
    ]
    ablation_metrics_exist = [m for m in ablation_metrics if m in results_summary.index]
    if ablation_metrics_exist:
        ablation_results = results_summary.loc[ablation_metrics_exist]
        for idx, row in ablation_results.iterrows():
            print(f"  {idx:<30}: {row['Mean']:.4f} ± {row['Std']:.4f}")
    else: print("  No ablation metrics to display.")
        
    print("\n---  Baselines (Mean ± Std) ---")
    baseline_metrics = [
        'BASELINE_Persistence_Overall_MAE', 'BASELINE_Persistence_Overall_RMSE',
        'BASELINE_Climatology_Overall_MAE', 'BASELINE_Climatology_Overall_RMSE'
    ]
    baseline_metrics_exist = [m for m in baseline_metrics if m in results_summary.index]
    if baseline_metrics_exist:
        baseline_results = results_summary.loc[baseline_metrics_exist]
        for idx, row in baseline_results.iterrows():
            print(f"  {idx:<30}: {row['Mean']:.4f} ± {row['Std']:.4f}")
    else: print("  No baseline metrics to display.")

    print("\n---  Complexity & Latency (Mean ± Std) ---")
    if all_run_complexity:
        avg_complexity = {}
        for run_comps in all_run_complexity:
            for model_name, comps in run_comps.items():
                if model_name not in avg_complexity:
                    avg_complexity[model_name] = {'params': [], 'latency_ms_per_sample': []}
                avg_complexity[model_name]['params'].append(comps.get('params', np.nan))
                avg_complexity[model_name]['latency_ms_per_sample'].append(comps.get('latency_ms_per_sample', np.nan))
        final_comp_df = pd.DataFrame({
            model: {
                'Params_Mean': np.nanmean(comps['params']),
                'Params_Std': np.nanstd(comps['params']),
                'Latency_ms_Mean': np.nanmean(comps['latency_ms_per_sample']),
                'Latency_ms_Std': np.nanstd(comps['latency_ms_per_sample'])
            } for model, comps in avg_complexity.items()
        }).T
        print(final_comp_df.to_markdown(floatfmt=".2f"))
    else: print("  No complexity data to display.")
    results_summary.to_csv(f"{DATASET_NAME}_ALL_RUNS_METRICS.csv")
    print(f"\n Full aggregated metrics saved to '{DATASET_NAME}_ALL_RUNS_METRICS.csv'")

    # --- 11. STATISTICAL SIGNIFICANCE TESTS ---
    print("\n" + "="*50)
    print("     STATISTICAL SIGNIFICANCE TESTS (T-TEST) 🔬")
    print("="*50 + "\n")
    print("  Comparing Ensemble MAE vs. Tuned Base Model MAEs")
    print("  (H0: Models are equal. p < 0.05 = H0 rejected, Ensemble is significantly better)\n")
    
    try:
        ens_mae = metrics_df['MAIN_ENSEMBLE_Overall_MAE']
        tcn_mae = metrics_df['ABL_TCN_Gated_Overall_MAE']
        lstm_mae = metrics_df['ABL_LSTM_Gated_Overall_MAE']
        lgbm_mae = metrics_df['ABL_LGBM_Gated_Overall_MAE']

        t_stat_tcn, p_val_tcn = ttest_rel(ens_mae, tcn_mae, alternative='less')
        t_stat_lstm, p_val_lstm = ttest_rel(ens_mae, lstm_mae, alternative='less')
        t_stat_lgbm, p_val_lgbm = ttest_rel(ens_mae, lgbm_mae, alternative='less')
        
        print(f"  Ensemble MAE: {ens_mae.mean():.4f} ± {ens_mae.std():.4f}")
        print(f"  TCN MAE:      {tcn_mae.mean():.4f} ± {tcn_mae.std():.4f}")
        print(f"  LSTM MAE:     {lstm_mae.mean():.4f} ± {lstm_mae.std():.4f}")
        print(f"  LGBM MAE:     {lgbm_mae.mean():.4f} ± {lgbm_mae.std():.4f}\n")
        
        print(f"  EnV vs. TCN-BiGRU: p-value = {p_val_tcn:.6f} ({'SIGNIFICANT' if p_val_tcn < 0.05 else 'NOT SIGNIFICANT'})")
        print(f"  EnV vs. BiLSTM:    p-value = {p_val_lstm:.6f} ({'SIGNIFICANT' if p_val_lstm < 0.05 else 'NOT SIGNIFICANT'})")
        print(f"  EnV vs. LGBM:      p-value = {p_val_lgbm:.6f} ({'SIGNIFICANT' if p_val_lgbm < 0.05 else 'NOT SIGNIFICANT'})")

    except Exception as e:
        print(f"   Could not perform statistical tests: {e}")
        print("  (This requires N_RUNS >= 2 to calculate variance)")

else:
    print(" No successful runs completed. No results to aggregate.")